## Création d'un df avec PySpark

In [1]:
# import des modules necessaires
from pyspark.sql import SparkSession

# ici nous allons crée une session Spark en donnant des configuration à notre cluster
spark = SparkSession.builder \
    .appName("Test") \
    .master("local[4]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

# Création de données et d'un DataFrame
data = [("Alice", 25), ("Bob", 30), ("Charlie", 35)]
df = spark.createDataFrame(data, ["nom", "age"])
df.show()

26/02/16 13:57:54 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
                                                                                

+-------+---+
|    nom|age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
+-------+---+



## Chargement d'un fichier csv avec PySpark

In [9]:
# Télécharge le fichier iris.data et le sauvegarde dans ~/spark-workspace/data/iris.csv
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data -O ~/spark-workspace/data/iris.csv

--2026-02-16 14:37:29--  https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘/home/tarik/spark-workspace/data/iris.csv’

/home/tarik/spark-w     [ <=>                ]   4,44K  --.-KB/s    in 0s      

2026-02-16 14:37:30 (95,1 MB/s) - ‘/home/tarik/spark-workspace/data/iris.csv’ saved [4551]



In [26]:
# Lire le CSV
df = spark.read.csv(
    "/home/tarik/spark-workspace/data/iris.csv",
    header=False,
    inferSchema=True
)

# Renommer les colonnes
noms_colonnes = ["sepal_length", "sepal_width", "petal_length", "petal_width", "class"]
df = df.toDF(*noms_colonnes)

df.show()

+------------+-----------+------------+-----------+-----------+
|sepal_length|sepal_width|petal_length|petal_width|      class|
+------------+-----------+------------+-----------+-----------+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|
|         5.4|        3.9|         1.7|        0.4|Iris-setosa|
|         4.6|        3.4|         1.4|        0.3|Iris-setosa|
|         5.0|        3.4|         1.5|        0.2|Iris-setosa|
|         4.4|        2.9|         1.4|        0.2|Iris-setosa|
|         4.9|        3.1|         1.5|        0.1|Iris-setosa|
|         5.4|        3.7|         1.5|        0.2|Iris-setosa|
|         4.8|        3.4|         1.6|        0.2|Iris-setosa|
|         4.8|        3.0|         1.4| 

In [27]:
# Statistiques
df.describe().show()

+-------+------------------+-------------------+------------------+------------------+--------------+
|summary|      sepal_length|        sepal_width|      petal_length|       petal_width|         class|
+-------+------------------+-------------------+------------------+------------------+--------------+
|  count|               150|                150|               150|               150|           150|
|   mean| 5.843333333333335| 3.0540000000000007|3.7586666666666693|1.1986666666666672|          NULL|
| stddev|0.8280661279778637|0.43359431136217375| 1.764420419952262|0.7631607417008414|          NULL|
|    min|               4.3|                2.0|               1.0|               0.1|   Iris-setosa|
|    max|               7.9|                4.4|               6.9|               2.5|Iris-virginica|
+-------+------------------+-------------------+------------------+------------------+--------------+



In [28]:
# Salaire moyen par département
from pyspark.sql.functions import avg, max, min, count

df.groupBy("class") \
    .agg(
        count("*").alias("class"),
        avg("petal_length").alias("petal_length_avg"),
        max("petal_width").alias("petal_width_max")
    ) \
    .orderBy("petal_length_avg", ascending=False) \
    .show()

+---------------+-----+----------------+---------------+
|          class|class|petal_length_avg|petal_width_max|
+---------------+-----+----------------+---------------+
| Iris-virginica|   50|           5.552|            2.5|
|Iris-versicolor|   50|            4.26|            1.8|
|    Iris-setosa|   50|           1.464|            0.6|
+---------------+-----+----------------+---------------+



## Volume de données très important

In [35]:
spark.sparkContext.setJobDescription("Volume de données très important")

# Générer 100 millions de lignes
df_big = spark.range(0, 100000000).toDF("id")

# Ajouter des colonnes
from pyspark.sql.functions import col, rand, when

df_big = df_big \
    .withColumn("valeur", (rand() * 1000).cast("int")) \
    .withColumn("categorie", when(col("valeur") < 300, "Faible")
                              .when(col("valeur") < 700, "Moyen")
                              .otherwise("Élevé"))

# Compter
print(f"Total lignes: {df_big.count():,}")

# Agréger
df_big.groupBy("categorie").count().show()

# C'est rapide grâce à Spark ! ⚡

Total lignes: 100,000,000
+---------+--------+
|categorie|   count|
+---------+--------+
|    Moyen|40008306|
|    Élevé|29993082|
|   Faible|29998612|
+---------+--------+



In [33]:
# En haut du notebook
monitor = SparkJobMonitor(spark)

# Puis pour chaque test
monitor.execute_and_monitor(lambda: df.count(), "Mon test")

# À la fin
monitor.show_history()

NameError: name 'SparkJobMonitor' is not defined